# Phase 1: Data Engineering & Knowledge Base Construction
**Objective**: Transform raw theological PDF into a high-fidelity Knowledge Base (Vector + Graph) and generate synthetic data for router distillation.

In [ ]:
import re
import json
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# Load the PDF document
pdf_path = "Compendium-of-the-Catechism-of-the-Catholic-Church.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("PDF loaded successfully!")

# Merge all page contents into a single string
full_text = "\n".join([doc.page_content for doc in documents])

In [ ]:
# extract Q&A section
start_marker = "1. What is the plan of God for man?"
end_marker = "APPENDIX" # page 151 where the main content ends and appendices begin 

# find the start 
main_start = full_text.find(start_marker)
content = full_text[main_start:] if main_start != -1 else full_text

# find the end 
appendix_start = content.find(end_marker)
if appendix_start != -1:
    content = content[:appendix_start]

# extract Q&A pairs using regex
qa_pattern = re.compile(r"(\d+)\.\s+([^?\n][^?]*\?)(.*?)(?=\s*\d+\.\s+|$)", re.DOTALL)

matches = qa_pattern.findall(content)
qa_dict = {}

for match in matches:
    q_id = int(match[0])
    # Process only if ID is between 1 and 598 and not already stored
    if 1 <= q_id <= 598 and q_id not in qa_dict:
        question = match[1].strip()
        answer_raw = match[2].strip()
        
        # Answer cleaning logic
        # 1. Remove 'glyph' special characters and unnecessary reference text
        answer_clean = re.sub(r"APPE'glyph\d+DIX.*", "", answer_raw) # 부록 잔재 제거
        answer_clean = re.sub(r"B\)\s+FORMULAS.*", "", answer_clean) # 교리 공식 잔재 제거
        answer_clean = re.sub(r"'glyph\d+[A-Z]*", "", answer_clean) # 깨진 글리프 문자 제거
        
        # 2. Collapse consecutive spaces and newlines into one
        answer_clean = re.sub(r"\s+", " ", answer_clean).strip()
        
        qa_dict[q_id] = {
            "id": q_id,
            "question": question,
            "answer": answer_clean
        }

# Sort by ID order
sorted_keys = sorted(qa_dict.keys())
qa_list = [qa_dict[k] for k in sorted_keys]

# Check for missing IDs to ensure data integrity
missing_ids = set(range(1, 599)) - set(qa_dict.keys())
if missing_ids:
    print(f"⚠️ Still missing IDs: {sorted(list(missing_ids))}")

In [ ]:
# Load the dataset
with open("compendium_qa_clean.json", "r", encoding="utf-8") as f:
    qa_list = json.load(f)

# Regex for text pruning (Sanitization)
# Targets residual headers (Section, Chapter, Creed, etc.) that appear 
# after standard sentence terminators (., ?, ", )) followed by a space.
cutoff_pattern = re.compile(
    r"(?<=[.!?\"\'\)])\s+(Section\s+(One|Two|Three|Four)|CHAPTER\s+[a-zA-Z]+|Part\s+(One|Two|Three|Four)|The Creed\b|Symbolum|The Apostles['’]? Creed)", 
    re.IGNORECASE
)

# Prune residual tags
cut_count = 0
for item in qa_list:
    answer = item['answer']
    match = cutoff_pattern.search(answer)
    
    if match:
        # Truncate the string at the start of the match and strip whitespace
        item['answer'] = answer[:match.start()].strip()
        cut_count += 1

# Save the finalized dataset
with open("compendium_qa_clean.json", "w", encoding="utf-8") as f:
    json.dump(qa_list, f, indent=4, ensure_ascii=False)

print(f"✅ Successfully pruned unnecessary chapter tags from {cut_count} items.")

In [ ]:
import os
import pickle
from neo4j import GraphDatabase
from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from tqdm import tqdm
from getpass import getpass

In [ ]:
# Ensure OpenAI API key is set in environment variables
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

# Neo4j configuration 
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
if "NEO4J_PASSWORD" not in os.environ:
    os.environ["NEO4J_PASSWORD"] = getpass("Enter your Neo4j Password: ")
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# Connect to Neo4j and OpenAI
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
client = OpenAI()
print("Neo4j and OpenAI clients initialized successfully!")

In [4]:
json_path = "compendium_qa_clean.json"
with open(json_path, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

In [ ]:
def fix_ocr_range(token):
    # 1. if there's no '1' or the token is too short, return as is (likely not a range)
    if '1' not in token or len(token) < 3:
        return token
    
    valid_splits = []
    # 2. explore all possible splits at '1' positions
    for i in range(1, len(token) - 1):
        if token[i] == '1':
            left = token[:i]
            right = token[i+1:]
            try:
                # 3. if the split forms a valid range (left < right and difference < 150), consider it
                if int(left) < int(right) and (int(right) - int(left)) < 150:
                    valid_splits.append({
                        'str': f"{left}-{right}",
                        'diff': int(right) - int(left),
                        'len_diff': abs(len(left) - len(right)) # di
                    })
            except ValueError:
                pass
    
    # 4. if there are valid splits, choose the one with the smallest length difference (most balanced) and then smallest numeric difference
    if valid_splits:
        valid_splits.sort(key=lambda x: (x['len_diff'], x['diff']))
        return valid_splits[0]['str']
        
    return token

# function to extract references from answer text
def extract_references(answer_text):
    match = re.match(r'^[\d\s,]+', answer_text)
    if match:
        raw_str = match.group().strip()
        raw_refs = re.split(r'[\s,]+', raw_str)
        fixed_refs = [fix_ocr_range(ref) for ref in raw_refs if ref]
        return fixed_refs
    return []

# extract references and create Document objects with metadata
chunks = []
for item in qa_data:
    combined_text = f"Question ID {item['id']}: {item['question']}\nAnswer: {item['answer']}"
    
    refs = extract_references(item['answer'])
    
    metadata = {"id": item['id']}
    if refs:
        metadata["refs"] = refs 
    
    doc = Document(
        page_content=combined_text,
        metadata=metadata
    )
    chunks.append(doc)

print(f"✅ {len(chunks)} chunks created with metadata.")

# verification
test_ids = [1, 11, 12, 21] 
for doc in chunks:
    if doc.metadata['id'] in test_ids:
        print(f"No. {doc.metadata['id']:>2}: {doc.metadata}")

In [ ]:
# Create embeddings and build vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("Vector DB created and saved successfully!")

In [ ]:
# Example similarity search
query = "What is the sacrament of Confirmation?"
docs = vectorstore.similarity_search(query, k=3)

print("query:", query)
for i, doc in enumerate(docs):
    print(f"\n--- Similarity Result {i+1} ---")
    print(doc.page_content)

In [ ]:
# Function to extract graph data using GPT-4o with strict rules and JSON output
def extract_graph_data(text_chunk, question_id):
    system_prompt = """
    You are an expert Catholic theologian and a strict Knowledge Graph Data Engineer. 
    Your task is to extract highly structured entities (nodes) and their relationships (edges) from the provided theological text to enable complex, multi-hop reasoning.

    # STRICT EXTRACTION RULES

    1. ONTOLOGY (ALLOWED TYPES ONLY - CRITICAL):
       - Node Types: [Person, Sacrament, Concept, Event, Document, Organization, Virtue, Prayer, CCC_Reference]
       - Relationship Types (Hierarchical): [IS_A, PART_OF, SUBSET_OF]
       - Relationship Types (Functional): [INSTITUTED_BY, REQUIRES, RESULTS_IN, SYMBOLIZES, ACCOMPLISHES, LEADS_TO, TRANSMITTED_BY, ACHIEVED_BY]
       - Relationship Types (Referential): [REFERENCES, DEFINES, EXPLAINS, RELATED_TO]
       * STRICT RULE: DO NOT invent new node or relationship types. If you cannot find an exact match for a relationship, use "RELATED_TO".

    2. ENTITY RESOLUTION & CANONICALIZATION:
       - Map pronouns and theological variants to their MOST STANDARD, canonical noun form.
         (e.g., "The Lord" / "Christ" -> "Jesus_Christ", "Holy Mass" -> "Eucharist", "The Church" -> "Catholic_Church").
       - Entity IDs MUST be concise (maximum 3 words) and formatted in PascalCase or snake_case.

    3. DESCRIPTIONS (MANDATORY):
       - You MUST generate a brief, context-specific `description` (1-2 sentences) for EVERY node based on the text. 
       - Do not leave the `description` field empty.

    4. EXTRACTING CCC REFERENCES (CROSS-REFERENCES):
       - Scan the text for numeric paragraph references (e.g., "39-43", "48", "cf. 215", "74").
       - Create a 'CCC_Reference' node for them with the ID format "CCC_Number" (e.g., "CCC_39-43", "CCC_74").
       - Always connect the main concept of the text to this CCC_Reference node using the 'REFERENCES' relationship.

    5. METADATA:
       - You MUST include the exact `source_id` (which is the Question ID provided in the user prompt) as a property inside EVERY node.

    # REQUIRED JSON OUTPUT FORMAT:
    {
      "nodes": [ 
        {"id": "Divine_Revelation", "type": "Concept", "source_id": "11", "description": "God's communication of himself to humanity, transmitted through Apostolic Tradition."},
        {"id": "CCC_74", "type": "CCC_Reference", "source_id": "11", "description": "Catechism paragraph referencing God's desire for all men to be saved."}
      ],
      "edges": [ 
        {"source": "Divine_Revelation", "relation": "TRANSMITTED_BY", "target": "Apostolic_Tradition"},
        {"source": "Divine_Revelation", "relation": "REFERENCES", "target": "CCC_74"} 
      ]
    }
    """
    
    response = client.chat.completions.create(
        model="gpt-4o", 
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            # include the question ID explicitly in the prompt to ensure source_id is not missed 
            {"role": "user", "content": f"Extract graph data for Question ID {question_id} from this text:\n\n{text_chunk}"}
        ],
        temperature=0.0 # creativity suppression, consistency maximization
    )
    
    return json.loads(response.choices[0].message.content)

In [ ]:
# Function to insert extracted graph data into Neo4j with source_id in both nodes and edges
def insert_graph_to_neo4j(graph_data, current_source_id):
    with driver.session() as session:
        # 1. generate nodes
        for node in graph_data.get("nodes", []):
            query = f"""
            MERGE (n:`{node['type']}` {{id: $id}})
            SET n.source_id = $source_id,
                n.description = $description
            """
            session.run(query, id=node['id'], source_id=node.get('source_id', current_source_id), description=node.get('description', ''))

        # 2. generate edges 
        for edge in graph_data.get("edges", []):
            query = f"""
            MATCH (a {{id: $source}})
            MATCH (b {{id: $target}})
            MERGE (a)-[r:`{edge['relation']}`]->(b)
            SET r.source_id = $source_id 
            """
            session.run(query, source=edge['source'], target=edge['target'], source_id=current_source_id)

print("✅ Neo4j storage function is ready!")

In [ ]:
# test the entire pipeline with a sample chunk
sample_chunk = chunks[10]
sample_text = sample_chunk.page_content 
current_id = sample_chunk.metadata.get('id')

print(f"Original Text [{current_id}]\n{sample_text}\n")

# extract graph data using GPT-4o
print("[Knowledge graph data extracted by GPT-4o]")
graph_data = extract_graph_data(sample_text, current_id)

print(json.dumps(graph_data, indent=2, ensure_ascii=False))

In [ ]:
# insert the extracted graph data into Neo4j
successful_chunks = 0
failed_chunks = 0

print("Starting the full extraction and insertion process for all chunks...")

# iterate through all chunks 
for i, chunk in enumerate(tqdm(chunks)): 
    try:
        # extract the question ID from the chunk metadata
        current_id = chunk.metadata.get('id')

        # extract graph data 
        graph_data = extract_graph_data(chunk.page_content, current_id)
        
        # save to Neo4j
        insert_graph_to_neo4j(graph_data, current_id)
        
        successful_chunks += 1
        
    except Exception as e:
        print(f"\n🚨 error in {i}th chunk (ID: {chunk.metadata.get('id')}) processing: {e}")
        failed_chunks += 1
        # continue to the next chunk even if there's an error in the current one
        continue

print(f"\n✨ data extraction and Neo4j insertion completed for all chunks!")
print(f"✅ successful: {successful_chunks}")
print(f"❌ failed: {failed_chunks}")

In [ ]:
# find the problematic chunk with ID 485 and attempt a manual fix
target_id = 485
problematic_chunk = next((c for c in chunks if c.metadata.get('id') == target_id), None)

if problematic_chunk:
    print(f"🔍 retrying ID {target_id}...")
    clean_content = re.sub(r'[^\x00-\x7F]+', ' ', problematic_chunk.page_content) 
    
    try:
        graph_data = extract_graph_data(clean_content, target_id)
        insert_graph_to_neo4j(graph_data, target_id)
        print(f"✅ successfully ID {target_id} restored and saved!")
        
    except Exception as e:
        print(f"❌ error still occurs: {e}")
        print("💡 tip: this question needs to be manually entered or the answer content might be too long, causing a token limit issue.")
else:
    print("Oops, couldn't find the chunk with ID 485!")

In [ ]:
# function to generate synthetic queries based on the text chunk and its CCC references
def generate_synthetic_queries(text_chunk, ccc_refs, question_id):
    system_prompt = """
    You are an expert AI Data Engineer creating a training dataset for an Adaptive RAG Router model.
    Based on the provided Catholic theology text, generate 3 realistic user queries. 
    Each query MUST fall into one of the exact categories below, aligning with our query taxonomy (Factual, Comparative, Interpretive, Application).

    # CATEGORIES & RULES
    1. "Vector_Search" (Factual Intent): A direct, simple question asking for a definition, fact, or summary explicitly stated in the text.
    2. "Graph_Search" (Comparative/Interpretive Intent): A complex question asking about the *relationship*, *hierarchy*, *cause-and-effect*, or *process* between multiple entities mentioned in the text. 
    3. "Out_of_Scope" (Negative Sample): A tricky question that uses keywords from the text but asks about modern politics, personal opinions, or historical events NOT covered in this specific text.

    # REQUIRED OUTPUT FORMAT (JSON ONE-SHOT EXAMPLE)
    {
      "dataset": [
        {
          "query": "What is the explicit definition of Apostolic Tradition?",
          "intent": "Vector_Search",
          "taxonomy": "Factual",
          "rationale": "Asks for a direct factual definition present in the text."
        },
        {
          "query": "How does Jesus Christ's command relate to the transmission of Divine Revelation?",
          "intent": "Graph_Search",
          "taxonomy": "Interpretive",
          "rationale": "Requires connecting the entity 'Jesus Christ' with the concept of 'Divine Revelation'."
        },
        {
          "query": "Which modern political party best aligns with Apostolic Tradition?",
          "intent": "Out_of_Scope",
          "taxonomy": "None",
          "rationale": "Uses the keyword but asks for modern political alignment, which is out of scope."
        }
      ]
    }
    """
    
    ref_info = f"\nRelated CCC References: {ccc_refs}" if ccc_refs else ""
    user_content = f"Question ID: {question_id}\nText:\n{text_chunk}{ref_info}"

    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ],
        temperature=0.7 
    )
    
    return json.loads(response.choices[0].message.content)

print("✅ function ready!\n")

In [ ]:
# test the synthetic query generation function with a sample chunk
test_results = []

print("🔥 Testing extraction started (5 chunks)...")

for i, chunk in enumerate(chunks[:5]):
    q_id = chunk.metadata.get('id')
    ccc_refs = chunk.metadata.get('refs', [])
    
    try:
        # strengthen the prompt by also passing the question ID to provide clearer context for the generated queries
        result = generate_synthetic_queries(chunk.page_content, ccc_refs, q_id)
        
        for data in result.get("dataset", []):
            data["source_id"] = q_id
            data["ccc_refs"] = ccc_refs
            test_results.append(data)
            
    except Exception as e:
        print(f"🚨 Error on ID {q_id}: {e}")

print(json.dumps(test_results[:6], indent=2, ensure_ascii=False))

In [ ]:
all_synthetic_data = []
failed_ids = []

print("🚀 generating synthetic training data...")

for chunk in tqdm(chunks):
    q_id = chunk.metadata.get('id')
    ccc_refs = chunk.metadata.get('refs', [])
    
    try:
        result = generate_synthetic_queries(chunk.page_content, ccc_refs, q_id)
        
        if "dataset" in result:
            for data in result["dataset"]:
                data["source_id"] = q_id
                data["ccc_refs"] = ccc_refs
                all_synthetic_data.append(data)
            
    except Exception as e:
        print(f"\n🚨 ID {q_id}에서 에러 발생: {e}")
        failed_ids.append(q_id)

# save the synthetic dataset to a JSON file
with open("synthetic_train_dataset.json", "w", encoding="utf-8") as f:
    json.dump(all_synthetic_data, f, indent=4, ensure_ascii=False)

print(f"\n✨ Done! In total, {len(all_synthetic_data)} synthetic training samples have been generated.")
if failed_ids:
    print(f"⚠️ Failed Question IDs: {failed_ids}")

In [ ]:
# retry the three failed questions
failed_ids = [399, 539, 597]
recovered_data = []

print(f"🛠️ starting to retry {len(failed_ids)} failed questions...")

target_chunks = [c for c in chunks if c.metadata.get('id') in failed_ids]

for chunk in target_chunks:
    q_id = chunk.metadata.get('id')
    ccc_refs = chunk.metadata.get('refs', [])
    
    try:
        print(f"Retrying ID {q_id}")
        result = generate_synthetic_queries(chunk.page_content, ccc_refs, q_id)
        
        if "dataset" in result:
            for data in result["dataset"]:
                data["source_id"] = q_id
                data["ccc_refs"] = ccc_refs
                recovered_data.append(data)
        print(f"✅ Success! ID {q_id} has been recovered!")
            
    except Exception as e:
        print(f"🚨 ID {q_id} still failed: {e}")

# save the recovered data and merge with existing dataset
if recovered_data:
    with open("synthetic_train_dataset.json", "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    existing_data.extend(recovered_data)

    with open("synthetic_train_dataset.json", "w", encoding="utf-8") as f:
        json.dump(existing_data, f, indent=4, ensure_ascii=False)
        
    print(f"\n✨ Recovery and merging completed! Now we have a perfect dataset with {len(existing_data)} samples and no missing data 💎")
else:
    print("\n😢 still failed to recover some questions.")